# Two-Tower Cold Start Ablation Study (v4 Architecture)
**CS 7180 — Applied Deep Learning**

Architecture v4: four sub-towers with fusion MLPs.

**Restaurant side:**
- **Content tower** (static): categories, price tier, structured attributes — what the restaurant IS.
- **Context tower** (static per-restaurant): checkin-derived temporal profile (24-bin hour distribution + 7-bin day-of-week distribution) — WHEN the restaurant is popular.
- **Restaurant fusion MLP**: learns cross-concern interactions (e.g. "Italian + weekend brunch pattern").

**User side:**
- **Content tower** (slow-changing): cuisine/food-type preference vector + simulated onboarding selections — WHO the user is.
- **Context tower** (per-query): day-of-week (cyclical sin/cos + is_weekend) + haversine distance + checkin-matched time-of-visit — the user's CURRENT SITUATION.
- **User fusion MLP**: learns cross-concern interactions.

**Cold-start robustness:**
- Input-level temporal dropout (20%) zeros `temporal_vec` during training so the full pipeline learns to produce reasonable restaurant embeddings without checkin data.
- Cold-restaurant evaluation zeros temporal for ALL candidates (fair content-only comparison).
- Cold-user onboarding uses leave-one-out protocol to prevent information leakage.

**New in this round:**
- **User checkin time-of-visit** features: hour-of-day and day-of-week sin/cos from checkin-matched review timestamps (~31% match rate). Complementary to the always-available `DayOfWeekGroup`.
- **Gated fusion MLP** (EmerG-inspired): content embedding produces per-dimension gates that control how context is mixed in, enabling item-specific fusion behavior.

This notebook evaluates the ablation study results from `run_ablation.py`.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch, Patch
import matplotlib.ticker as mtick

# ── Configuration ──────────────────────────────────────────────────────

RESULTS_DIR = Path("../results/ablation")

# v4 architecture + user checkin + gated fusion experiments
EXPERIMENTS = {
    # Core ablations
    "full_model":              ("Full Model",            "Content + Context", "Content + Context",  "MLP"),
    "no_temporal":             ("No Temporal",            "Content Only",      "Content + Context",  "MLP"),
    "no_restaurant_content":   ("No Rest. Content",       "Context Only",      "Content + Context",  "MLP"),
    "no_user_context":         ("No User Context",        "Content + Context", "Content Only",       "MLP"),
    "no_user_content":         ("No User Content",        "Content + Context", "Context Only",       "MLP"),
    "no_onboarding":           ("No Onboarding",          "Content + Context", "Ctx + Prefs",        "MLP"),
    "no_preferences":          ("No Preferences",         "Content + Context", "Ctx + Onboard",      "MLP"),
    "content_x_content":       ("Content \u00d7 Content", "Content Only",      "Content Only",       "MLP"),
    "context_x_context":       ("Context \u00d7 Context", "Content + Context", "Context Only",       "MLP"),
    # User checkin ablations
    "with_user_checkin":       ("Full + Checkin",         "Content + Context", "Content + Context+", "MLP"),
    "checkin_replaces_dow":    ("Checkin replaces DoW",   "Content + Context", "Content + Checkin",  "MLP"),
    # Gated fusion ablations
    "gated_fusion":            ("Gated Fusion",           "Content + Context", "Content + Context",  "Gated"),
    "gated_fusion_with_checkin":("Gated + Checkin",       "Content + Context", "Content + Context+", "Gated"),
}

SPLITS = ["warm", "cold_restaurant", "cold_user"]
SPLIT_LABELS = {"warm": "Warm", "cold_restaurant": "Cold Restaurant", "cold_user": "Cold User"}
SPLIT_COLORS = {"warm": "#3498db", "cold_restaurant": "#e74c3c", "cold_user": "#f39c12"}

# Display order groups experiments logically
DISPLAY_ORDER = [
    # Core model
    "full_model",
    # Restaurant-side ablations
    "no_temporal", "no_restaurant_content",
    # User-side ablations
    "no_user_context", "no_user_content",
    # Cross-concern
    "content_x_content", "context_x_context",
    # User content breakdown
    "no_onboarding", "no_preferences",
    # User checkin
    "with_user_checkin", "checkin_replaces_dow",
    # Gated fusion
    "gated_fusion", "gated_fusion_with_checkin",
]

# ── Load eval results ─────────────────────────────────────────────────

rows = []
for exp_name, (label, rest_type, user_type, fusion_type) in EXPERIMENTS.items():
    path = RESULTS_DIR / exp_name / "eval_results.json"
    if not path.exists():
        continue
    data = json.load(open(path))
    for split in SPLITS:
        tt = data.get(split, {}).get("TwoTower", {})
        bl_rand = data.get(split, {}).get("Random", {})
        bl_pop = data.get(split, {}).get("Popularity", {})
        rows.append({
            "experiment": exp_name,
            "label": label,
            "restaurant_tower": rest_type,
            "user_tower": user_type,
            "fusion": fusion_type,
            "split": split,
            "split_label": SPLIT_LABELS[split],
            "Hit@5": tt.get("Hit@5"),
            "NDCG@10": tt.get("NDCG@10"),
            "n_cases": tt.get("n_cases"),
            "Random_Hit@5": bl_rand.get("Hit@5"),
            "Random_NDCG@10": bl_rand.get("NDCG@10"),
            "Popularity_Hit@5": bl_pop.get("Hit@5"),
            "Popularity_NDCG@10": bl_pop.get("NDCG@10"),
        })

df = pd.DataFrame(rows)
n_loaded = df["experiment"].nunique()
n_total = len(EXPERIMENTS)
print(f"Loaded {n_loaded}/{n_total} experiments")
if n_loaded < n_total:
    missing = set(EXPERIMENTS.keys()) - set(df["experiment"].unique())
    print(f"  Missing: {', '.join(sorted(missing))}")
print(f"  Splits: {', '.join(SPLIT_LABELS[s] for s in SPLITS)}")

## 1. Training Dynamics
Parse `train.log` from each experiment to extract per-epoch loss, Hit@5, and NDCG@10 convergence curves.

In [ ]:
LOG_PATTERN = re.compile(
    r"Epoch\s+(\d+)/\d+\s*\|\s*Loss:\s*([\d.]+)\s*\|\s*"
    r"Hit@5:\s*([\d.]+)\s*\|\s*NDCG@10:\s*([\d.]+)"
)

log_rows = []
for exp_name in EXPERIMENTS:
    log_path = RESULTS_DIR / exp_name / "train.log"
    if not log_path.exists():
        continue
    text = log_path.read_text()
    for m in LOG_PATTERN.finditer(text):
        log_rows.append({
            "experiment": exp_name,
            "label": EXPERIMENTS[exp_name][0],
            "epoch": int(m.group(1)),
            "loss": float(m.group(2)),
            "hit5": float(m.group(3)),
            "ndcg10": float(m.group(4)),
        })

log_df = pd.DataFrame(log_rows)
if len(log_df) > 0:
    print(f"Parsed training logs for {log_df['experiment'].nunique()} experiments")
    print(f"  Epoch range: {log_df['epoch'].min()} \u2013 {log_df['epoch'].max()}")
else:
    print("No training logs found \u2014 skipping convergence plots")

In [ ]:
if len(log_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    exp_list = [e for e in DISPLAY_ORDER if e in log_df["experiment"].values]
    cmap = plt.colormaps["tab20"] if len(exp_list) > 10 else plt.colormaps["tab10"]
    color_map = {e: cmap(i / max(len(exp_list) - 1, 1)) for i, e in enumerate(exp_list)}

    for ax, (col, ylabel, title) in zip(axes, [
        ("loss", "BPR Loss", "Training Loss"),
        ("hit5", "Hit@5 (Val)", "Validation Hit@5"),
        ("ndcg10", "NDCG@10 (Val)", "Validation NDCG@10"),
    ]):
        for exp_name in exp_list:
            edf = log_df[log_df["experiment"] == exp_name]
            lw = 2.5 if exp_name == "full_model" else 1.2
            ls = "-" if exp_name in ("full_model", "with_user_checkin", "gated_fusion") else "--"
            ax.plot(edf["epoch"], edf[col], label=EXPERIMENTS[exp_name][0],
                    color=color_map[exp_name], lw=lw, ls=ls, alpha=0.85)

        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.grid(alpha=0.3)

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.22, 0.5),
               fontsize=7, framealpha=0.9)
    fig.suptitle("Training Convergence Across Ablations", fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("Skipping convergence plots (no log data)")

## 2. Dataset & Split Statistics
Test case counts and baseline performance per split.

In [ ]:
stats_rows = []
for split in SPLITS:
    sdf = df[(df["split"] == split) & (df["experiment"] == "full_model")]
    if sdf.empty:
        sdf = df[df["split"] == split].iloc[:1]
    if sdf.empty:
        continue
    row = sdf.iloc[0]
    stats_rows.append({
        "Split": SPLIT_LABELS[split],
        "Test Cases": int(row["n_cases"]) if pd.notna(row["n_cases"]) else "\u2014",
        "Random Hit@5": f"{row['Random_Hit@5']:.4f}" if pd.notna(row["Random_Hit@5"]) else "\u2014",
        "Random NDCG@10": f"{row['Random_NDCG@10']:.4f}" if pd.notna(row["Random_NDCG@10"]) else "\u2014",
        "Popularity Hit@5": f"{row['Popularity_Hit@5']:.4f}" if pd.notna(row["Popularity_Hit@5"]) else "\u2014",
        "Popularity NDCG@10": f"{row['Popularity_NDCG@10']:.4f}" if pd.notna(row["Popularity_NDCG@10"]) else "\u2014",
    })

stats_df = pd.DataFrame(stats_rows).set_index("Split")
print("Evaluation Split Statistics")
print("=" * 80)
display(stats_df)

print("\nExpected baselines on 1-positive + 99-negatives (100 candidates):")
print("  Random Hit@5 \u2248 5.0% (uniform chance of top-5 containing the positive)")
print("  Popularity = 0 on cold restaurants (zero training reviews by definition)")

## 3. Full Results Table
All experiments × splits × metrics. Best values per split are highlighted.

In [ ]:
summary = df.pivot_table(
    index="label", columns=["split_label"],
    values=["Hit@5", "NDCG@10", "n_cases"],
).round(4)

col_order = ["Warm", "Cold Restaurant", "Cold User"]
summary = summary.reindex(columns=col_order, level=1)
row_order = [EXPERIMENTS[e][0] for e in DISPLAY_ORDER if e in df["experiment"].values]
summary = summary.reindex(row_order)

display(
    summary[["Hit@5", "NDCG@10"]]
    .style
    .highlight_max(axis=0, color="#d4edda")
    .highlight_min(axis=0, color="#f8d7da")
    .format("{:.4f}")
    .set_caption("Two-Tower Ablation Results (best=green, worst=red)")
)

## 4. Per-Split Bar Charts
Side-by-side Hit@5 and NDCG@10 for all experiments with baseline reference lines.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8), sharey=True)

for ax, split in zip(axes, SPLITS):
    split_df = df[df["split"] == split].set_index("experiment").reindex(
        [e for e in DISPLAY_ORDER if e in df["experiment"].values]
    )
    labels = split_df["label"]
    x = np.arange(len(labels))
    w = 0.35

    bars_h = ax.bar(x - w / 2, split_df["Hit@5"], w, label="Hit@5", color="#3498db", alpha=0.85)
    bars_n = ax.bar(x + w / 2, split_df["NDCG@10"], w, label="NDCG@10", color="#e67e22", alpha=0.85)

    for bars in [bars_h, bars_n]:
        for bar in bars:
            h = bar.get_height()
            if pd.notna(h) and h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.003,
                        f"{h:.3f}", ha="center", va="bottom", fontsize=5, rotation=90)

    pop = split_df["Popularity_Hit@5"].iloc[0]
    rand = split_df["Random_Hit@5"].iloc[0]
    if pd.notna(pop) and pop > 0:
        ax.axhline(pop, ls="--", color="gray", alpha=0.6, label=f"Popularity H@5 ({pop:.3f})")
    if pd.notna(rand):
        ax.axhline(rand, ls=":", color="gray", alpha=0.4, label=f"Random H@5 ({rand:.3f})")

    ax.set_title(SPLIT_LABELS[split], fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7)
    ax.grid(axis="y", alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel("Score")
    ax.legend(fontsize=6, loc="upper right")

fig.suptitle("Ablation Results by Split", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 5. Feature Contribution (ΔHit@5 vs Full Model)
Negative delta = removing this feature group hurts performance. Positive = it was actually *hurting* the full model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
ablations = [e for e in DISPLAY_ORDER if e != "full_model"]

for ax, split in zip(axes, SPLITS):
    split_df = df[df["split"] == split].set_index("experiment")
    if "full_model" not in split_df.index:
        continue
    full_h5 = split_df.loc["full_model", "Hit@5"]

    abl_df = split_df.loc[[e for e in ablations if e in split_df.index]]
    deltas = abl_df["Hit@5"] - full_h5
    colors = ["#e74c3c" if d < 0 else "#2ecc71" for d in deltas]

    y = np.arange(len(deltas))
    ax.barh(y, deltas, color=colors, edgecolor="white", height=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels([EXPERIMENTS[e][0] for e in deltas.index], fontsize=8)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{SPLIT_LABELS[split]} \u2014 \u0394Hit@5 vs Full", fontsize=12, fontweight="bold")
    ax.set_xlabel("\u0394Hit@5")
    ax.grid(axis="x", alpha=0.3)

    for i, (val, name) in enumerate(zip(deltas, deltas.index)):
        offset = 0.003 if val >= 0 else -0.003
        ha = "left" if val >= 0 else "right"
        ax.text(val + offset, i, f"{val:+.3f}", va="center", ha=ha, fontsize=8)

fig.suptitle("Feature Group Contribution (\u0394Hit@5 vs Full Model)",
             fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

## 6. User Checkin Ablation
Does checkin-matched time-of-visit improve over review-date day-of-week? And is it additive or substitutive?

In [ ]:
checkin_exps = ["full_model", "with_user_checkin", "checkin_replaces_dow"]
checkin_labels = {
    "full_model": "Full Model\n(DoW only)",
    "with_user_checkin": "Full + Checkin\n(DoW + Checkin)",
    "checkin_replaces_dow": "Checkin replaces DoW\n(Checkin only)",
}

cdf = df[df["experiment"].isin(checkin_exps)].copy()
if len(cdf) > 0 and cdf["experiment"].nunique() >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, split in zip(axes, SPLITS):
        sdf = cdf[cdf["split"] == split].set_index("experiment").reindex(
            [e for e in checkin_exps if e in cdf["experiment"].values]
        )
        labels = [checkin_labels.get(e, e) for e in sdf.index]
        x = np.arange(len(labels))
        w = 0.35

        ax.bar(x - w / 2, sdf["Hit@5"], w, label="Hit@5", color="#3498db", alpha=0.85)
        ax.bar(x + w / 2, sdf["NDCG@10"], w, label="NDCG@10", color="#e67e22", alpha=0.85)

        for xi, (h5, n10) in enumerate(zip(sdf["Hit@5"], sdf["NDCG@10"])):
            if pd.notna(h5):
                ax.text(xi - w / 2, h5 + 0.005, f"{h5:.3f}", ha="center", fontsize=8)
            if pd.notna(n10):
                ax.text(xi + w / 2, n10 + 0.005, f"{n10:.3f}", ha="center", fontsize=8)

        ax.set_title(SPLIT_LABELS[split], fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=7, ha="center")
        ax.grid(axis="y", alpha=0.3)
        if ax == axes[0]:
            ax.set_ylabel("Score")
            ax.legend(fontsize=8)

    fig.suptitle("User Checkin Time-of-Visit: Additive vs Substitutive",
                 fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("User checkin experiments not found \u2014 skipping")

## 7. Fusion Architecture Comparison
Standard MLP fusion vs content-gated fusion (EmerG-inspired). The gated fusion lets each item's content control how context is mixed in.

In [ ]:
fusion_pairs = [
    ("full_model", "gated_fusion", "Base (no checkin)"),
    ("with_user_checkin", "gated_fusion_with_checkin", "With checkin"),
]

fusion_exps = set()
for mlp_e, gated_e, _ in fusion_pairs:
    fusion_exps.update([mlp_e, gated_e])
fdf = df[df["experiment"].isin(fusion_exps)]

if fdf["experiment"].nunique() >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, split in zip(axes, SPLITS):
        sdf = fdf[fdf["split"] == split].set_index("experiment")
        x = np.arange(len(fusion_pairs))
        w = 0.35

        mlp_vals = [sdf.loc[mlp_e, "Hit@5"] if mlp_e in sdf.index else 0 for mlp_e, _, _ in fusion_pairs]
        gated_vals = [sdf.loc[gated_e, "Hit@5"] if gated_e in sdf.index else 0 for _, gated_e, _ in fusion_pairs]
        pair_labels = [label for _, _, label in fusion_pairs]

        ax.bar(x - w / 2, mlp_vals, w, label="MLP Fusion", color="#3498db", alpha=0.85)
        ax.bar(x + w / 2, gated_vals, w, label="Gated Fusion", color="#9b59b6", alpha=0.85)

        for xi, (mv, gv) in enumerate(zip(mlp_vals, gated_vals)):
            if mv > 0:
                ax.text(xi - w / 2, mv + 0.005, f"{mv:.3f}", ha="center", fontsize=8)
            if gv > 0:
                ax.text(xi + w / 2, gv + 0.005, f"{gv:.3f}", ha="center", fontsize=8)

        ax.set_title(SPLIT_LABELS[split], fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(pair_labels, fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        if ax == axes[0]:
            ax.set_ylabel("Hit@5")
            ax.legend(fontsize=8)

    fig.suptitle("MLP vs Gated Fusion: Hit@5 by Split",
                 fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("Gated fusion experiments not found \u2014 skipping")

## 8. User Content Ablations
Isolating the individual contribution of **preferences** vs **onboarding** on the user side.

In [ ]:
targeted = ["full_model", "no_onboarding", "no_preferences", "no_user_content"]
targeted_labels = {
    "full_model": "Full\n(Pref + Onboard + Ctx)",
    "no_onboarding": "No Onboarding\n(Pref + Ctx)",
    "no_preferences": "No Preferences\n(Onboard + Ctx)",
    "no_user_content": "No User Content\n(Ctx Only)",
}

tdf = df[df["experiment"].isin(targeted)].copy()
if len(tdf) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, split in zip(axes, SPLITS):
        sdf = tdf[tdf["split"] == split].set_index("experiment").reindex(
            [e for e in targeted if e in tdf["experiment"].values]
        )
        labels = [targeted_labels.get(e, e) for e in sdf.index]
        x = np.arange(len(labels))
        w = 0.35

        ax.bar(x - w / 2, sdf["Hit@5"], w, label="Hit@5", color="#3498db", alpha=0.85)
        ax.bar(x + w / 2, sdf["NDCG@10"], w, label="NDCG@10", color="#e67e22", alpha=0.85)

        for xi, (h5, n10) in enumerate(zip(sdf["Hit@5"], sdf["NDCG@10"])):
            if pd.notna(h5):
                ax.text(xi - w / 2, h5 + 0.005, f"{h5:.3f}", ha="center", fontsize=8)
            if pd.notna(n10):
                ax.text(xi + w / 2, n10 + 0.005, f"{n10:.3f}", ha="center", fontsize=8)

        ax.set_title(SPLIT_LABELS[split], fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=7, ha="center")
        ax.grid(axis="y", alpha=0.3)
        if ax == axes[0]:
            ax.set_ylabel("Score")
            ax.legend(fontsize=8)

    fig.suptitle("User Content Ablation: Preferences vs Onboarding",
                 fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("Targeted ablation experiments not found \u2014 skipping")

## 9. Cross-Split Profile
How does each experiment's performance shift across warm vs cold splits?

In [ ]:
exp_present = [e for e in DISPLAY_ORDER if e in df["experiment"].values]

fig, ax = plt.subplots(figsize=(14, 7))

cmap = plt.colormaps["tab20"] if len(exp_present) > 10 else plt.colormaps["tab10"]
color_map = {e: cmap(i / max(len(exp_present) - 1, 1)) for i, e in enumerate(exp_present)}

x_positions = np.arange(len(SPLITS))

for exp_name in exp_present:
    edf = df[df["experiment"] == exp_name].set_index("split")
    vals = [edf.loc[s, "Hit@5"] if s in edf.index else np.nan for s in SPLITS]

    lw = 2.5 if exp_name == "full_model" else 1.3
    marker = "o" if exp_name in ("full_model", "with_user_checkin", "gated_fusion") else "s"
    ms = 8 if exp_name == "full_model" else 5

    ax.plot(x_positions, vals, marker=marker, markersize=ms,
            label=EXPERIMENTS[exp_name][0], color=color_map[exp_name],
            lw=lw, alpha=0.85)

ax.set_xticks(x_positions)
ax.set_xticklabels([SPLIT_LABELS[s] for s in SPLITS], fontsize=11)
ax.set_ylabel("Hit@5", fontsize=12)
ax.set_title("Cross-Split Performance Profile (Hit@5)", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, framealpha=0.9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Feature Importance Summary
Per-feature ΔHit@5 when ablated, organized by which side (restaurant vs user) and concern (content vs context).

In [ ]:
tower_impact = []

for split in SPLITS:
    sdf = df[df["split"] == split].set_index("experiment")
    if "full_model" not in sdf.index:
        continue
    full = sdf.loc["full_model", "Hit@5"]

    for exp, feature_label, side in [
        ("no_temporal",           "Restaurant Temporal",    "Restaurant"),
        ("no_restaurant_content", "Restaurant Content",     "Restaurant"),
        ("no_user_context",       "User Context (all)",     "User"),
        ("no_user_content",       "User Content (all)",     "User"),
        ("no_onboarding",         "Onboarding",             "User"),
        ("no_preferences",        "Preferences",            "User"),
    ]:
        if exp in sdf.index:
            val = sdf.loc[exp, "Hit@5"]
            tower_impact.append({
                "split": split,
                "split_label": SPLIT_LABELS[split],
                "feature": feature_label,
                "side": side,
                "delta": val - full,
            })

    # Additive features (positive delta = improvement over full)
    for exp, feature_label, side in [
        ("with_user_checkin", "+ User Checkin", "User (additive)"),
    ]:
        if exp in sdf.index:
            val = sdf.loc[exp, "Hit@5"]
            tower_impact.append({
                "split": split,
                "split_label": SPLIT_LABELS[split],
                "feature": feature_label,
                "side": side,
                "delta": val - full,
            })

impact_df = pd.DataFrame(tower_impact)

if len(impact_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)
    side_colors = {"Restaurant": "#e74c3c", "User": "#3498db", "User (additive)": "#2ecc71"}

    for ax, split in zip(axes, SPLITS):
        sdf = impact_df[impact_df["split"] == split]
        colors = [side_colors.get(s, "gray") for s in sdf["side"]]
        bars = ax.barh(range(len(sdf)), sdf["delta"], color=colors, alpha=0.8)
        ax.set_yticks(range(len(sdf)))
        ax.set_yticklabels(sdf["feature"], fontsize=9)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlabel("\u0394Hit@5 vs Full Model")
        ax.set_title(SPLIT_LABELS[split], fontsize=12, fontweight="bold")
        ax.invert_yaxis()

        for i, val in enumerate(sdf["delta"]):
            offset = 0.002 if val >= 0 else -0.002
            ha = "left" if val >= 0 else "right"
            ax.text(val + offset, i, f"{val:+.4f}", va="center", ha=ha, fontsize=8)

    legend_elements = [Patch(facecolor=c, alpha=0.8, label=s) for s, c in side_colors.items()]
    fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10,
               bbox_to_anchor=(0.5, -0.02))

    fig.suptitle("Feature Importance by Side (\u0394Hit@5 vs Full Model)",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("No ablation data available")

In [ ]:
# ── Full Landscape: Lollipop dot plot (all configs × 3 splits) ───────

landscape = df[df["split"].isin(SPLITS)].pivot_table(
    index="experiment", columns="split", values="Hit@5"
).reindex(columns=SPLITS)

# Add baselines
for bl, col in [("Random", "Random_Hit@5"), ("Popularity", "Popularity_Hit@5")]:
    bl_vals = {}
    for split in SPLITS:
        row = df[(df["experiment"] == "full_model") & (df["split"] == split)]
        bl_vals[split] = row[col].values[0] if len(row) > 0 else None
    landscape.loc[bl] = bl_vals

landscape = landscape.sort_values("cold_restaurant", ascending=True)

label_map = {k: v[0] for k, v in EXPERIMENTS.items()}
label_map["Random"] = "Random"
label_map["Popularity"] = "Popularity"
landscape.index = [label_map.get(e, e) for e in landscape.index]

fig, ax = plt.subplots(figsize=(14, 9))
y = np.arange(len(landscape))

for i, (split, color, marker) in enumerate([
    ("warm", "#3498db", "o"),
    ("cold_restaurant", "#e74c3c", "D"),
    ("cold_user", "#f39c12", "s"),
]):
    vals = landscape[split].values
    for j, v in enumerate(vals):
        if pd.notna(v):
            ax.plot([0, v], [j, j], color=color, alpha=0.15, linewidth=1.5)
    ax.scatter(vals, y, color=color, s=80, marker=marker,
               label=SPLIT_LABELS[split], zorder=3, edgecolors="white", linewidth=0.5)

# Gray bands for baselines
for label_name in ["Random", "Popularity"]:
    if label_name in landscape.index:
        idx = list(landscape.index).index(label_name)
        ax.axhspan(idx - 0.4, idx + 0.4, color="#eeeeee", zorder=0)

# ── Annotations ──────────────────────────────────────────────────────

# Vertical reference line at random expected Hit@5
ax.axvline(x=0.05, color="#9e9e9e", linestyle=":", alpha=0.5, zorder=0)
ax.text(0.052, len(landscape) - 0.3, "Random\nfloor (5%)",
        fontsize=8, color="#777", va="top")

# Bracket / label for top Cold Restaurant performers
top_cr = landscape["cold_restaurant"].nlargest(3)
top_y_min = list(landscape.index).index(top_cr.index[-1])
top_y_max = list(landscape.index).index(top_cr.index[0])
ax.annotate("Best cold restaurant\nconfigurations",
            xy=(top_cr.iloc[0] + 0.003, top_y_max),
            fontsize=9, fontweight="bold", color="#c0392b",
            va="center")

# Label the warm/cold-user cluster on the right
ax.annotate("Warm & Cold User\nperformance clusters\ntightly (0.25–0.32)",
            xy=(0.32, len(landscape) * 0.35),
            fontsize=9, color="#555", style="italic",
            va="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#ccc", alpha=0.9))

# Callout for Popularity at zero on cold restaurant
pop_idx = list(landscape.index).index("Popularity")
ax.annotate("Popularity = 0.00\n(no review history)",
            xy=(0.001, pop_idx),
            xytext=(0.12, pop_idx - 0.8),
            fontsize=8, color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2),
            va="center")

ax.set_yticks(y)
ax.set_yticklabels(landscape.index, fontsize=10)
ax.set_xlabel("Hit@5", fontsize=12)
ax.set_title("Full Ablation Landscape — Hit@5 by Split", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.set_xlim(-0.01, max(landscape.max()) + 0.04)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Key Findings (Auto-Generated)

In [ ]:
from IPython.display import display, Markdown

def lookup(exp, split, metric="Hit@5"):
    mask = (df["experiment"] == exp) & (df["split"] == split)
    vals = df.loc[mask, metric]
    return float(vals.iloc[0]) if len(vals) > 0 and pd.notna(vals.iloc[0]) else None

def lookup_baseline(split, baseline, metric="Hit@5"):
    col = f"{baseline}_{metric}"
    mask = df["split"] == split
    vals = df.loc[mask, col].dropna()
    return float(vals.iloc[0]) if len(vals) > 0 else None

def make_table(title, question, condition_rows, interp_fn):
    """Build a markdown comparison table with auto-interpretation."""
    header = f"### {title}\n**Question:** {question}\n\n"
    header += "| Condition | " + " | ".join(f"{SPLIT_LABELS[s]} H@5" for s in SPLITS)
    header += " | " + " | ".join(f"{SPLIT_LABELS[s]} lift" for s in SPLITS) + " |\n"
    header += "|" + "---|" * (1 + 2 * len(SPLITS)) + "\n"

    table_rows = []
    for label, exp_name in condition_rows:
        parts = [label]
        for split in SPLITS:
            val = lookup(exp_name, split)
            parts.append(f"{val:.4f}" if val is not None else "\u2014")
        for split in SPLITS:
            val = lookup(exp_name, split)
            baseline = "Popularity" if split != "cold_restaurant" else "Random"
            bl = lookup_baseline(split, baseline)
            if val is not None and bl is not None and bl > 0:
                parts.append(f"{val/bl:.2f}\u00d7")
            else:
                parts.append("\u2014")
        table_rows.append("| " + " | ".join(parts) + " |")

    body = "\n".join(table_rows)
    interp = interp_fn(condition_rows)
    return header + body + f"\n\n**Interpretation:** {interp}\n"


# ── Finding 0: Full landscape ────────────────────────────────────────

def interp_landscape(rows):
    parts = []
    for split in SPLITS:
        vals = {exp: lookup(exp, split) for _, exp in rows if lookup(exp, split) is not None}
        if not vals:
            continue
        best_exp = max(vals, key=vals.get)
        worst_exp = min(vals, key=vals.get)
        best_label = next(l for l, e in rows if e == best_exp)
        worst_label = next(l for l, e in rows if e == worst_exp)
        bl_name = "Popularity" if split != "cold_restaurant" else "Random"
        bl = lookup_baseline(split, bl_name)
        parts.append(
            f"On {SPLIT_LABELS[split]}, best is {best_label} ({vals[best_exp]:.4f})"
            + (f" at {vals[best_exp]/bl:.2f}\u00d7" if bl and bl > 0 else "")
            + f"; worst is {worst_label} ({vals[worst_exp]:.4f})."
        )
    return " ".join(parts) if parts else "Insufficient data."

all_conditions = [(EXPERIMENTS[e][0], e) for e in DISPLAY_ORDER if e in df["experiment"].values]
# Add baselines
for bl_name in ["Random", "Popularity"]:
    all_conditions.insert(0, (bl_name, bl_name))

# Patch lookup to handle baseline names
_orig_lookup = lookup
def lookup(exp, split, metric="Hit@5"):
    if exp in ("Random", "Popularity"):
        return lookup_baseline(split, exp, metric)
    return _orig_lookup(exp, split, metric)

f0 = make_table(
    "Finding 0: Full Landscape \u2014 Baselines vs All Configurations",
    "How does every ablation configuration (including user checkin and gated fusion) compare against non-personalized baselines?",
    all_conditions,
    interp_landscape,
)

# Restore original lookup
lookup = _orig_lookup


# ── Finding 1: Restaurant temporal ────────────────────────────────────

def interp_temporal(rows):
    parts = []
    for split in SPLITS:
        full = lookup("full_model", split)
        no_temp = lookup("no_temporal", split)
        if full is not None and no_temp is not None:
            delta = full - no_temp
            sl = SPLIT_LABELS[split]
            parts.append(f"On {sl}, temporal contributes \u0394={delta:+.4f} ({full:.4f} \u2192 {no_temp:.4f} without it).")
    if parts:
        full_cr = lookup("full_model", "cold_restaurant")
        no_temp_cr = lookup("no_temporal", "cold_restaurant")
        if full_cr is not None and no_temp_cr is not None:
            delta = full_cr - no_temp_cr
            if abs(delta) < 0.005:
                parts.append("Minimal cold-restaurant impact is expected: temporal is zeroed for all candidates in that split anyway.")
            elif delta > 0:
                parts.append("Temporal helps cold-restaurant ranking \u2014 the model uses temporal training signal to learn better content representations.")
            else:
                parts.append("Temporal hurts cold-restaurant performance \u2014 the model may be over-relying on temporal signal.")
    return " ".join(parts) if parts else "Insufficient data."

f1 = make_table(
    "Finding 1: Restaurant Temporal Contribution",
    "Does the restaurant's checkin temporal profile help recommendations, and does it hurt cold-start restaurants?",
    [("Full Model", "full_model"), ("No Temporal", "no_temporal")],
    interp_temporal,
)


# ── Finding 2: User context vs content ────────────────────────────────

def interp_user(rows):
    parts = []
    for split in SPLITS:
        full = lookup("full_model", split)
        no_ctx = lookup("no_user_context", split)
        no_cnt = lookup("no_user_content", split)
        if all(v is not None for v in [full, no_ctx, no_cnt]):
            ctx_delta = full - no_ctx
            cnt_delta = full - no_cnt
            sl = SPLIT_LABELS[split]
            winner = "context" if abs(ctx_delta) > abs(cnt_delta) else "content"
            parts.append(f"On {sl}, user {winner} matters more (ctx \u0394={ctx_delta:+.4f}, cnt \u0394={cnt_delta:+.4f}).")
    return " ".join(parts) if parts else "Insufficient data."

f2 = make_table(
    "Finding 2: User Tower \u2014 Context vs Content",
    "Does user context (dow, distance) or content (preferences, onboarding) contribute more?",
    [
        ("Full Model", "full_model"),
        ("No User Context", "no_user_context"),
        ("No User Content", "no_user_content"),
    ],
    interp_user,
)


# ── Finding 3: Preferences vs onboarding ──────────────────────────────

def interp_pref_vs_onboard(rows):
    parts = []
    full = {s: lookup("full_model", s) for s in SPLITS}
    no_onb = {s: lookup("no_onboarding", s) for s in SPLITS}
    no_pref = {s: lookup("no_preferences", s) for s in SPLITS}
    no_both = {s: lookup("no_user_content", s) for s in SPLITS}

    if all(full[s] is not None and no_onb[s] is not None and no_pref[s] is not None for s in SPLITS):
        onb_delta_w = full["warm"] - no_onb["warm"]
        pref_delta_w = full["warm"] - no_pref["warm"]
        if abs(onb_delta_w) > abs(pref_delta_w):
            winner, loser = "Onboarding", "preferences"
        else:
            winner, loser = "Preferences", "onboarding"
        parts.append(
            f"{winner} has a larger warm impact than {loser} "
            f"(\u0394={max(abs(onb_delta_w), abs(pref_delta_w)):+.4f} vs {min(abs(onb_delta_w), abs(pref_delta_w)):+.4f})."
        )

        onb_delta_cu = full["cold_user"] - no_onb["cold_user"]
        pref_delta_cu = full["cold_user"] - no_pref["cold_user"]
        if no_both["cold_user"] is not None and no_both["cold_user"] >= full["cold_user"]:
            parts.append(
                f"On cold users, neither signal helps \u2014 context-only ({no_both['cold_user']:.4f}) "
                f"matches or beats the full model ({full['cold_user']:.4f})."
            )
        elif abs(onb_delta_cu) > abs(pref_delta_cu):
            parts.append(f"On cold users, onboarding has more impact (\u0394={onb_delta_cu:+.4f}).")
        else:
            parts.append(f"On cold users, preferences have more impact (\u0394={pref_delta_cu:+.4f}).")

    return " ".join(parts) if parts else "Targeted ablation data not available."

f3 = make_table(
    "Finding 3: Preferences vs Onboarding",
    "Which user content feature contributes more?",
    [
        ("Full Model", "full_model"),
        ("No Onboarding", "no_onboarding"),
        ("No Preferences", "no_preferences"),
        ("No User Content", "no_user_content"),
    ],
    interp_pref_vs_onboard,
)


# ── Finding 4: Cross-concern interactions ─────────────────────────────

def interp_cross_concern(rows):
    parts = []
    for split in SPLITS:
        full = lookup("full_model", split)
        cxc = lookup("content_x_content", split)
        ctx_ctx = lookup("context_x_context", split)
        if all(v is not None for v in [full, cxc, ctx_ctx]):
            sl = SPLIT_LABELS[split]
            cxc_delta = full - cxc
            ctx_delta = full - ctx_ctx
            parts.append(
                f"On {sl}, removing all context costs \u0394={cxc_delta:+.4f}; "
                f"removing all content costs \u0394={ctx_delta:+.4f}."
            )
    if parts:
        cxc_cr = lookup("content_x_content", "cold_restaurant")
        full_cr = lookup("full_model", "cold_restaurant")
        if cxc_cr is not None and full_cr is not None:
            if abs(cxc_cr - full_cr) < 0.01:
                parts.append("Content\u00d7Content matches full model on cold restaurants, confirming context is zeroed correctly.")
    return " ".join(parts) if parts else "Insufficient data."

f4 = make_table(
    "Finding 4: Cross-Concern Interactions (Content\u00d7Content vs Context\u00d7Context)",
    "How much do the fusion MLPs benefit from having both content and context?",
    [
        ("Full Model", "full_model"),
        ("Content \u00d7 Content", "content_x_content"),
        ("Context \u00d7 Context", "context_x_context"),
    ],
    interp_cross_concern,
)


# ── Finding 5: User checkin time-of-visit ─────────────────────────────

def interp_checkin(rows):
    parts = []
    for split in SPLITS:
        full = lookup("full_model", split)
        with_ck = lookup("with_user_checkin", split)
        repl = lookup("checkin_replaces_dow", split)
        sl = SPLIT_LABELS[split]
        if full is not None and with_ck is not None:
            delta = with_ck - full
            parts.append(f"On {sl}, adding checkin: \u0394={delta:+.4f}.")
        if full is not None and repl is not None:
            delta = repl - full
            parts.append(f"Replacing DoW with checkin: \u0394={delta:+.4f}.")
    return " ".join(parts) if parts else "Checkin experiments not yet complete."

f5 = make_table(
    "Finding 5: User Checkin Time-of-Visit",
    "Does checkin-matched hour/day context improve over review-date day-of-week?",
    [
        ("Full Model (DoW only)", "full_model"),
        ("Full + Checkin (additive)", "with_user_checkin"),
        ("Checkin replaces DoW", "checkin_replaces_dow"),
    ],
    interp_checkin,
)


# ── Finding 6: Gated fusion ──────────────────────────────────────────

def interp_gated(rows):
    parts = []
    for split in SPLITS:
        full = lookup("full_model", split)
        gated = lookup("gated_fusion", split)
        sl = SPLIT_LABELS[split]
        if full is not None and gated is not None:
            delta = gated - full
            parts.append(f"On {sl}, gated fusion \u0394={delta:+.4f} vs MLP fusion.")
    for split in SPLITS:
        ck_mlp = lookup("with_user_checkin", split)
        ck_gated = lookup("gated_fusion_with_checkin", split)
        sl = SPLIT_LABELS[split]
        if ck_mlp is not None and ck_gated is not None:
            delta = ck_gated - ck_mlp
            parts.append(f"With checkin on {sl}, gated \u0394={delta:+.4f} vs MLP.")
    return " ".join(parts) if parts else "Gated fusion experiments not yet complete."

f6 = make_table(
    "Finding 6: Gated Fusion Architecture",
    "Does content-gated context mixing outperform standard MLP fusion?",
    [
        ("MLP Fusion (full)", "full_model"),
        ("Gated Fusion (full)", "gated_fusion"),
        ("MLP + Checkin", "with_user_checkin"),
        ("Gated + Checkin", "gated_fusion_with_checkin"),
    ],
    interp_gated,
)


# ── Render ────────────────────────────────────────────────────────────
display(Markdown(
    "## Key Findings (v4 Architecture)\n\n"
    "Auto-generated from ablation results. The v4 four-tower architecture has fusion MLPs "
    "on both sides, with restaurant context (temporal) on the restaurant tower. "
    "Input-level temporal dropout (20%) ensures the model can function without checkin data.\n\n---\n\n"
    + f0 + "\n\n" + f1 + "\n\n" + f2 + "\n\n" + f3 + "\n\n" + f4 + "\n\n" + f5 + "\n\n" + f6
))
